[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/raya-lucaria/ia_o26/blob/main/course/10_deep_learning/_assets/03_finetuning_lora.ipynb)

# Notebook 3 · Fine-tuning de imágenes con LoRA: tu dataset

Vas a enseñarle a **Stable Diffusion 1.5** el estilo de **tus imágenes** con un adaptador **LoRA**, y a medir con honestidad qué aprendió.

- **Entra:** tu `imagenes.zip`, en el formato de la tarea (`metadata.csv` + `images/` + `prompts.txt` + `LEEME.md`).
- **Sale:** imágenes antes y después, curvas de loss, cinco métricas explicadas, el examen en test y tu adaptador descargado (unos MB).
- **Sin zip también corre:** arma un dataset demo de caras de anime en el mismo formato y lo usa.

**Tres pasos**

1. «Entorno de ejecución → Cambiar tipo de entorno → **T4 GPU** → Guardar».
2. Arrastra tu `imagenes.zip` al panel **📁 Archivos** de la izquierda (detalles en la sección 02).
3. «Entorno de ejecución → **Ejecutar todo**».

**Cuánto tarda.** La primera celda de código fija `MODO`:

| `MODO` | Qué cambia | Tiempo en una T4 |
|---|---|---|
| `"corto"` (por omisión) | lo esencial | ≈ 20–25 min |
| `"largo"` | más pasos, más imágenes y más muestras para las métricas (lo marcado ➕) | ≈ 50–60 min |

**Test, una sola vez.** Todo lo que se mira para decidir usa **validation**. El conjunto de **test** se toca **una vez**, en la sección 08, con el adaptador ya entrenado.

**Mapa**

| § | Sección | Qué haces |
|---|---|---|
| 01 | Instalar y configurar | `MODO`, GPU, semillas, tamaños |
| 02 | Tu zip | subirlo, revisarlo, verlo, partirlo 80 / 10 / 10 |
| 03 | Antes | el modelo base con tus `prompts.txt` |
| 04 | LoRA por dentro | la fórmula y cuántos parámetros entrenas de verdad |
| 05 | Entrenar | un LoRA $r=8$ y sus curvas de train y validation |
| 06 | Ver LoRA | el rango de $\Delta W$, qué capas cambiaron, la perilla $\lambda$ |
| 07 | Métricas | qué mide cada una, su trampa, y su valor en validation |
| 08 | Testeo | test una vez, olvido, descargar el adaptador, qué mover si salió mal |

## 01 · Instalar y configurar

- **Elige el modo en la primera línea de código.** `MODO = "corto"` es el de clase; `MODO = "largo"` entrena más pasos y mide con más muestras.
- La celda instala solo lo que falta. No actualiza lo que Colab ya trae.

| Paquete | Para qué |
|---|---|
| `diffusers` | Stable Diffusion: UNet, VAE, schedulers |
| `transformers` | el text encoder y el modelo CLIP que mide las imágenes |
| `peft` | el adaptador LoRA |
| `accelerate` | carga eficiente de los pesos |

In [ ]:
MODO = "corto"   # "corto" (≈ 20–25 min en una T4) o "largo" (≈ 50–60 min)
LARGO = MODO == "largo"

# Instala solo lo que falta; no actualiza lo que Colab ya trae.
import importlib.util
import subprocess
import sys

FALTAN = [p for p in ["diffusers", "peft", "accelerate", "transformers"]
          if importlib.util.find_spec(p) is None]
if FALTAN:
    print("Instalando:", ", ".join(FALTAN))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *FALTAN],
                   check=True)
else:
    print("No falta nada que instalar.")
print("MODO =", MODO, "· LARGO =", LARGO)

- **Necesitas GPU.** Si aparece el aviso, elige T4 GPU en «Cambiar tipo de entorno» y ejecuta todo de nuevo.
- `DTYPE` es fp16 en la GPU: la mitad de memoria que fp32. La T4 no tiene bf16.
- `RAPIDO=1` existe solo para probar el notebook en una computadora sin GPU: usa modelos diminutos y sus imágenes son ruido. En Colab nunca está activo.

In [ ]:
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import diffusers
import peft
import transformers

# RAPIDO=1: prueba local en CPU con modelos diminutos (sus imágenes son ruido).
RAPIDO = os.environ.get("RAPIDO") == "1"
HAY_GPU = torch.cuda.is_available()
DEVICE = torch.device("cuda" if HAY_GPU and not RAPIDO else "cpu")
CUDA = DEVICE.type == "cuda"
DTYPE = torch.float16 if CUDA else torch.float32   # la T4 no tiene bf16

if not HAY_GPU and not RAPIDO:
    print("=" * 72)
    print("  ¡AVISO! ESTE ENTORNO NO TIENE GPU.")
    print("  Stable Diffusion en CPU tardaría horas. En Colab:")
    print("    Entorno de ejecución → Cambiar tipo de entorno → T4 GPU")
    print("    → Guardar, y luego Entorno de ejecución → Ejecutar todo.")
    print("=" * 72)

for nombre, mod in [("torch", torch), ("diffusers", diffusers),
                    ("peft", peft), ("transformers", transformers)]:
    print(f"{nombre:14s}{mod.__version__}")
print("DEVICE =", DEVICE, "· DTYPE =", DTYPE, "· RAPIDO =", RAPIDO)
if HAY_GPU:
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} · {props.total_memory / 2**30:.1f} GiB")

`P` junta todos los tamaños del modo. Los que más importan:

| Clave | Qué es |
|---|---|
| `res` | lado de las imágenes en px (SD 1.5 trabaja a 512) |
| `max_imagenes` | si tu zip trae más, se toma una muestra con semilla 0 |
| `rango`, `lr` | el rango $r$ del adaptador y el learning rate de AdamW |
| `batch`, `pasos` | imágenes por paso y número de pasos de entrenamiento |
| `val_cada` | cada cuántos pasos se mide la loss de validation |
| `semillas` | las semillas de las mallas: misma semilla ⇒ mismo ruido inicial |
| `n_gen` | cuántas imágenes se generan por conjunto para las métricas |

- `semilla(s)` fija el azar. Con la misma semilla, el modelo base y el adaptado parten del **mismo ruido**: la diferencia entre sus imágenes es efecto del adaptador.
- Colores fijos: azul train, naranja validation, verde test; gris el modelo base y morado el LoRA.

In [ ]:
# El modelo que ajustamos y el CLIP que mide las imágenes.
if RAPIDO:
    MODELO = "hf-internal-testing/tiny-stable-diffusion-pipe"
    MODELO_CLIP = "hf-internal-testing/tiny-random-CLIPModel"
else:
    MODELO = "stable-diffusion-v1-5/stable-diffusion-v1-5"
    MODELO_CLIP = "openai/clip-vit-base-patch32"

# Tamaños por modo. pasos = actualizaciones del adaptador.
if RAPIDO:
    P = dict(res=32, max_imagenes=60, demo_personajes=0, demo_por_personaje=0,
             rango=8, lr=1e-4, batch=2, pasos=4, val_cada=2, n_val_ruido=2,
             semillas=[0, 1], n_gen=3, pasos_inferencia=2)
elif LARGO:
    P = dict(res=512, max_imagenes=3000, demo_personajes=40,
             demo_por_personaje=15, rango=8, lr=1e-4, batch=4, pasos=1200,
             val_cada=100, n_val_ruido=16, semillas=[0, 1, 2, 3], n_gen=16,
             pasos_inferencia=30)
else:
    P = dict(res=512, max_imagenes=1000, demo_personajes=20,
             demo_por_personaje=10, rango=8, lr=1e-4, batch=4, pasos=400,
             val_cada=50, n_val_ruido=8, semillas=[0, 1], n_gen=8,
             pasos_inferencia=20)
display(pd.Series(P, name="valor").to_frame())

# Un color fijo por conjunto, como en los notebooks 1 y 2.
COLORES = {"train": "#1f77b4", "validation": "#ff7f0e", "test": "#2ca02c",
           "base": "#7f7f7f", "LoRA": "#9467bd"}


def semilla(s: int):
    """Fija las semillas de random, NumPy y PyTorch (CPU y GPU)."""
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)        # también siembra la GPU


semilla(0)
plt.style.use("default")
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3,
                     "figure.dpi": 80, "savefig.dpi": 80,
                     "axes.spines.top": False, "axes.spines.right": False})
try:                            # en Colab: figuras en JPEG, el notebook pesa menos
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("jpeg", pil_kwargs={"quality": 80})
except Exception:
    pass

## 02 · Tu zip

**Dónde subirlo.** Elige una de tres formas:

| Forma | Cómo | Cuándo |
|---|---|---|
| Panel de archivos | clic en **📁 Archivos** (barra de la izquierda) y **arrastra** tu `imagenes.zip` ahí | lo normal; espera a que termine la barrita de carga |
| Google Drive | quita el `#` de la línea `drive.mount(...)` y escribe la ruta en `RUTA_ZIP` | si tu zip pesa mucho |
| Botón de subir | quita el `#` de la línea `files.upload()` | si el panel no te deja arrastrar |

- Con `RUTA_ZIP = ""` la celda **busca sola** `imagenes*.zip` en `/content` (también `imagenes (1).zip`). Si hay varios, toma el más reciente.
- **Sin zip, el notebook no se detiene:** usa el dataset demo, que la celda siguiente arma en el mismo formato.
- Si encuentra un `texto.zip`, se detiene y te manda al notebook 4, que es el de texto.
- Los archivos subidos al panel **se borran** al cerrar la sesión de Colab. Tu zip original sigue en tu computadora.

In [ ]:
RUTA_ZIP = ""   # vacío: lo busco yo. O escribe la ruta, p. ej.
#                 "/content/drive/MyDrive/imagenes.zip"

# Alternativas (quita el # de la que quieras usar):
# from google.colab import files; files.upload()                 # un botón
# from google.colab import drive; drive.mount("/content/drive")  # tu Drive

from pathlib import Path

NB4 = ("https://colab.research.google.com/github/raya-lucaria/ia_o26/blob/"
       "main/course/10_deep_learning/_assets/04_finetuning_llm.ipynb")


def buscar(patron):
    """Zips que coinciden con `patron` en /content y en la carpeta actual,
    del más reciente al más viejo."""
    vistos = {p.resolve() for c in (Path("/content"), Path.cwd())
              if c.is_dir() for p in c.glob(patron) if p.is_file()}
    return sorted(vistos, key=lambda p: p.stat().st_mtime, reverse=True)


if RUTA_ZIP:
    if not Path(RUTA_ZIP).is_file():
        raise FileNotFoundError(f"No existe {RUTA_ZIP!r}. Revisa la ruta en el "
                                "panel 📁 (clic derecho → Copiar ruta).")
elif buscar("imagenes*.zip"):          # «imagenes (1).zip» también cuenta
    RUTA_ZIP = str(buscar("imagenes*.zip")[0])
elif buscar("texto*.zip"):
    raise ValueError(
        f"Encontré {buscar('texto*.zip')[0].name}: es un dataset de TEXTO. "
        f"Este notebook es el de imágenes. Abre el notebook 4:\n{NB4}")
ES_DEMO = not RUTA_ZIP
if ES_DEMO:
    otros = [p.name for p in buscar("*.zip")]
    if otros:
        print("Hay estos zip, pero ninguno se llama imagenes*.zip:", otros)
        print("Si uno es el tuyo, escribe su ruta en RUTA_ZIP y corre de nuevo.")
    print("No encontré tu imagenes.zip: uso el dataset DEMO (caras de anime).")
else:
    print("Uso tu zip:", RUTA_ZIP)

**El demo: caras de anime.** Si no subiste nada, esta celda arma `demo/imagenes.zip` **en el mismo formato que la tarea**, así que todo lo que sigue corre igual que con tu zip.

- Las caras vienen de la colección *moeimouto*, compilada en `ComputerVisionAnimeProject/AnimeFaceColorization` (CC-BY-4.0), en una versión fija del dataset.
- Personajes y caras se eligen con semilla 0. Todas llevan el mismo caption, `"an anime face"`: el modelo aprende un estilo, no identidades. Los derechos de los personajes son de sus autores; aquí solo se estudia el método.
- Las caras miden 252 px, menos que los 512 que pide la tarea: verás el aviso de imágenes chicas, y es cierto.
- Sus `prompts.txt` piden variaciones que no están en el dataset (pelo verde, lentes, un señor mayor).

In [ ]:
import csv
import re
import shutil
from concurrent.futures import ThreadPoolExecutor

from PIL import Image, ImageDraw

DATASET_DEMO = "ComputerVisionAnimeProject/AnimeFaceColorization"
REV_DEMO = "e8deff76ecee132a5eead1b03941789ba60098fa"   # versión fija
PROMPTS_DEMO = ["an anime face with short green hair",
                "an anime face wearing glasses",
                "an anime face, smiling, red eyes",
                "an anime face of an old man"]


def caras_reales():
    """N caras por personaje, personajes y caras elegidos con semilla 0."""
    from huggingface_hub import HfApi, hf_hub_download
    api = HfApi()

    def listar(carpeta):
        return [x.path for x in api.list_repo_tree(
            DATASET_DEMO, repo_type="dataset", path_in_repo=carpeta,
            revision=REV_DEMO)]

    def bajar(ruta):
        return Image.open(hf_hub_download(
            DATASET_DEMO, ruta, repo_type="dataset", revision=REV_DEMO))

    # Solo las carpetas reference/NNN_<personaje>: son las caras.
    personajes = sorted(r for r in listar("reference")
                        if re.match(r"^\d{3}_", r.split("/")[-1]))
    azar = random.Random(0)
    elegidos = sorted(azar.sample(personajes, P["demo_personajes"]))
    with ThreadPoolExecutor(16) as pool:
        rutas = []
        for lista in pool.map(listar, elegidos):
            pngs = sorted(r for r in lista if r.endswith(".png"))
            rutas += azar.sample(pngs, min(len(pngs), P["demo_por_personaje"]))
        return [im.convert("RGB") for im in pool.map(bajar, rutas)]


def caras_de_juguete(n=50):
    """Para la prueba en CPU: óvalos de colores, sin descargar nada."""
    azar, salida = random.Random(0), []
    for _ in range(n):
        im = Image.new("RGB", (64, 64), tuple(azar.choices(range(256), k=3)))
        ImageDraw.Draw(im).ellipse((12, 8, 52, 58),
                                   fill=tuple(azar.choices(range(256), k=3)))
        salida.append(im)
    return salida

In [ ]:
def armar_demo(carpeta=Path("demo/imagenes")):
    """Escribe el demo en el formato de la tarea y lo empaqueta en un zip."""
    shutil.rmtree(carpeta, ignore_errors=True)
    (carpeta / "images").mkdir(parents=True)
    caras = caras_de_juguete() if RAPIDO else caras_reales()
    with open(carpeta / "metadata.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["file_name", "text"])
        for i, im in enumerate(caras):
            im.save(carpeta / f"images/{i:04d}.jpg", quality=95)
            w.writerow([f"images/{i:04d}.jpg", "an anime face"])
    (carpeta / "prompts.txt").write_text("\n".join(PROMPTS_DEMO) + "\n",
                                         encoding="utf-8")
    (carpeta / "LEEME.md").write_text(
        "# Dataset demo\n"
        "- **Qué quiero que aprenda el modelo:** el estilo «cara de anime»\n"
        "- **Ruta:** imágenes\n"
        f"- **Fuente:** https://huggingface.co/datasets/{DATASET_DEMO}\n"
        "- **Licencia:** CC-BY-4.0 (las caras son de personajes con derechos"
        " de sus autores; solo para estudiar el método)\n"
        f"- **Cuántos ejemplos y cómo los elegí:** {len(caras)} caras, "
        "personajes y caras al azar con semilla 0\n", encoding="utf-8")
    return shutil.make_archive(str(carpeta), "zip", carpeta)


if ES_DEMO:
    RUTA_ZIP = armar_demo()
    print("Demo armado en", RUTA_ZIP)
else:
    print("(tienes tu zip: no hace falta el demo)")

**Revisar antes de usar.** `revisar_zip` es **la misma función** de la página de la tarea, copiada letra por letra. Si ya te dijo `OK` en tu computadora, aquí también.

- Si el zip es de **texto** (trae `datos.jsonl`), la celda se detiene y te da el enlace al notebook 4.
- Si algo falla, el error dice **qué** está mal (una columna, un archivo que falta, una imagen que no abre). Corrige el zip, súbelo otra vez y vuelve a correr desde aquí.
- El error más común: comprimir **la carpeta** en vez de los archivos. Entonces `metadata.csv` queda en `imagenes/metadata.csv` y no en la raíz.

In [ ]:
# Copia literal de la página de la tarea (sección 5).
import csv, io, json, zipfile
from PIL import Image

def revisar_zip(ruta):
    """Dice si el zip sigue el formato de la tarea. No cambia nada."""
    z = zipfile.ZipFile(ruta)
    nombres = set(z.namelist())
    faltan = {"prompts.txt", "LEEME.md"} - nombres
    assert not faltan, f"faltan en la raíz del zip: {sorted(faltan)}"
    prompts = z.read("prompts.txt").decode("utf-8").split("\n")
    prompts = [p for p in prompts if p.strip()]
    assert 3 <= len(prompts) <= 5, f"prompts.txt: {len(prompts)} prompts, van de 3 a 5"
    if "metadata.csv" in nombres:                     # imágenes
        filas = list(csv.DictReader(io.StringIO(z.read("metadata.csv").decode("utf-8"))))
        assert filas and set(filas[0]) == {"file_name", "text"}, "columnas: file_name,text"
        assert len(filas) >= 50, f"solo {len(filas)} imágenes; mínimo 50"
        for fila in filas:
            assert fila["file_name"] in nombres, f"no está {fila['file_name']}"
            assert fila["text"].strip(), f"{fila['file_name']} sin texto"
            Image.open(z.open(fila["file_name"])).verify()
        print(f"OK · imágenes: {len(filas)} · prompts: {len(prompts)}")
    elif "datos.jsonl" in nombres:                    # texto
        lineas = z.read("datos.jsonl").decode("utf-8").splitlines()
        ejemplos = [json.loads(l) for l in lineas if l.strip()]
        assert len(ejemplos) >= 200, f"solo {len(ejemplos)} ejemplos; mínimo 200"
        for i, ej in enumerate(ejemplos, 1):
            assert set(ej) == {"prompt", "completion"}, f"línea {i}: llaves {sorted(ej)}"
            assert ej["completion"].strip(), f"línea {i}: completion vacío"
        print(f"OK · ejemplos: {len(ejemplos)} · prompts: {len(prompts)}")
    else:
        raise AssertionError("falta metadata.csv (imágenes) o datos.jsonl (texto)")

In [ ]:
try:
    with zipfile.ZipFile(RUTA_ZIP) as z:
        NOMBRES_ZIP = z.namelist()
except zipfile.BadZipFile:
    raise ValueError(f"{RUTA_ZIP} no es un zip válido: vuelve a empaquetarlo "
                     "(página de la tarea, sección 5).") from None
if "metadata.csv" not in NOMBRES_ZIP and "datos.jsonl" in NOMBRES_ZIP:
    raise ValueError("Este zip es un dataset de TEXTO (trae datos.jsonl). "
                     f"Este notebook es el de imágenes. Abre el notebook 4:\n{NB4}")
try:
    revisar_zip(RUTA_ZIP)
except AssertionError as e:
    pista = ""
    if any(n.endswith("/metadata.csv") for n in NOMBRES_ZIP):
        pista = ("\nPista: metadata.csv está dentro de una carpeta. Comprime "
                 "los archivos, no la carpeta.")
    raise ValueError(f"Tu zip no sigue el formato de la tarea: {e}{pista}\n"
                     "Corrígelo con la página de la tarea (secciones 2 y 5) "
                     "y vuelve a subirlo.") from None

**De tu zip a tres conjuntos.**

| Paso | Qué hace | Por qué |
|---|---|---|
| Descomprimir | en `datos_zip/` | leer cada imagen y su caption |
| Muestra | si traes más de `P["max_imagenes"]`, toma una muestra con semilla 0 | que el modo corto quepa en clase |
| Recortar | endereza la foto, recorta un cuadrado **al centro** y lo deja a 512 × 512 | SD 1.5 trabaja a 512 px |
| Deduplicar | hash de la imagen reducida a 16 × 16 | una imagen repetida en train y en test haría que test premie memorizar |
| Partir | 80 % train, 10 % validation, 10 % test, con semilla 0 | misma partición en cada corrida |

- **Train** enseña. **Validation** sirve para mirar y decidir. **Test** se usa una sola vez, en la sección 08.
- Cada imagen conserva **su** caption (columna `text`). Si todas dicen lo mismo, verás «captions distintos = 1».

In [ ]:
import hashlib

from PIL import ImageOps

CARPETA_ZIP = Path("datos_zip")             # aquí se descomprime tu zip
shutil.rmtree(CARPETA_ZIP, ignore_errors=True)
with zipfile.ZipFile(RUTA_ZIP) as z:
    z.extractall(CARPETA_ZIP)
with open(CARPETA_ZIP / "metadata.csv", encoding="utf-8") as f:
    FILAS = list(csv.DictReader(f))
PROMPTS = [p.strip() for p in (CARPETA_ZIP / "prompts.txt")
           .read_text(encoding="utf-8").split("\n") if p.strip()]
LEEME = (CARPETA_ZIP / "LEEME.md").read_text(encoding="utf-8")
if len(FILAS) > P["max_imagenes"]:
    print(f"Tu zip trae {len(FILAS)} imágenes; tomo {P['max_imagenes']} "
          "al azar con semilla 0 (P['max_imagenes']).")
    FILAS = random.Random(0).sample(FILAS, P["max_imagenes"])


def abrir(fila):
    """Abre, endereza (EXIF), recorta al centro y deja a P["res"] px."""
    im = Image.open(CARPETA_ZIP / fila["file_name"])
    im.draft("RGB", (P["res"], P["res"]))   # un JPEG enorme se lee ya reducido
    im = ImageOps.exif_transpose(im).convert("RGB")
    lado = min(im.size)
    return ImageOps.fit(im, (P["res"], P["res"]), Image.BICUBIC), lado


def huella(im):
    """Hash de la imagen reducida a 16×16: detecta duplicados casi exactos."""
    return hashlib.md5(im.resize((16, 16), Image.BILINEAR).tobytes()).hexdigest()


with ThreadPoolExecutor(8) as pool:
    ABIERTAS = list(pool.map(abrir, FILAS))
# Deduplicar ANTES de partir: una imagen repetida en train y en test
# haría que test premie memorizar.
vistas, UNICAS = set(), []
for fila, (im, lado) in zip(FILAS, ABIERTAS):
    if huella(im) not in vistas:
        vistas.add(huella(im))
        UNICAS.append((im, fila["text"].strip(), lado))
chicas = sum(lado < 512 for _, _, lado in UNICAS)

# Partición 80 / 10 / 10 con semilla 0.
orden = list(range(len(UNICAS)))
random.Random(0).shuffle(orden)
n_val = n_test = max(1, round(0.1 * len(orden)))
PARTES = {"train": orden[n_val + n_test:], "validation": orden[:n_val],
          "test": orden[n_val:n_val + n_test]}
IMGS = {k: [UNICAS[i][0] for i in v] for k, v in PARTES.items()}
TEXTOS = {k: [UNICAS[i][1] for i in v] for k, v in PARTES.items()}
display(pd.DataFrame({k: {"imágenes": len(v),
                          "captions distintos": len(set(TEXTOS[k]))}
                      for k, v in IMGS.items()}))
print(f"duplicadas descartadas: {len(FILAS) - len(UNICAS)}")
if chicas and not RAPIDO:
    print(f"Ojo: {chicas} imágenes tienen el lado corto < 512 px; se agrandan "
          "y quedan más suaves. El modelo también aprende esa suavidad.")

- Hasta ocho imágenes de train, ya recortadas, con su caption encima. **Revisa que el recorte al centro no corte lo importante.**
- Debajo, tus prompts de prueba y tu ficha. Los prompts vuelven en la sección 03 (antes) y en la 08 (después).

In [ ]:
import math
import textwrap


def malla(imagenes, filas, titulo, etiquetas=None):
    """Malla de imágenes (≤ 256 px). etiquetas: una por imagen (título
    chico) o una por fila (a la izquierda)."""
    n = len(imagenes)
    cols = max(1, math.ceil(n / filas))
    fig, ejes = plt.subplots(filas, cols, figsize=(1.9 * cols, 2.0 * filas),
                             squeeze=False, dpi=80)
    for i, eje in enumerate(ejes.flat):
        eje.axis("off")
        if i >= n:
            continue
        im = imagenes[i].copy()
        im.thumbnail((256, 256))
        eje.imshow(im)
        if etiquetas is not None and len(etiquetas) == n:
            eje.set_title(textwrap.shorten(etiquetas[i], 28), fontsize=7)
    if etiquetas is not None and len(etiquetas) == filas and filas != n:
        for f in range(filas):
            ejes[f, 0].text(-0.05, 0.5, textwrap.fill(etiquetas[f], 22),
                            fontsize=8, ha="right", va="center",
                            transform=ejes[f, 0].transAxes)
    fig.suptitle(titulo, fontsize=10)
    plt.tight_layout()
    plt.show()


k = min(8, len(IMGS["train"]))
malla(IMGS["train"][:k], filas=1 if k <= 4 else 2,
      titulo=f"Imágenes de train (de {len(IMGS['train'])}), con su caption",
      etiquetas=TEXTOS["train"][:k])
print("Tus prompts de prueba (prompts.txt):")
for p in PROMPTS:
    print("  ·", p)
print("\nTu ficha (LEEME.md):\n" + LEEME)

## 03 · Antes: el modelo base

Antes de entrenar, mira qué hace Stable Diffusion **sin** tus datos. Sin este «antes» no hay forma de saber qué cambió.

Stable Diffusion tiene tres piezas, y **solo el UNet recibirá el adaptador**:

| Pieza | Entrada → salida | En este notebook |
|---|---|---|
| VAE | imagen 512 × 512 × 3 ↔ latent 64 × 64 × 4 | congelado |
| Text encoder (CLIP) | caption → embedding $c$ de 77 tokens | congelado |
| UNet $\varepsilon_\theta$ | $(x_t, t, c)$ → ruido estimado | congelado, **más el adaptador LoRA** |

- La celda baja el modelo (≈ 4 GB, uno o dos minutos la primera vez) y cuenta los parámetros de cada pieza.
- El *safety checker* va apagado para ahorrar memoria; tus datos y prompts son tu responsabilidad (lo que no se vale está en la página de la tarea).

In [ ]:
import gc

from diffusers import (AutoencoderKL, DPMSolverMultistepScheduler,
                       StableDiffusionPipeline, UNet2DConditionModel)
from transformers import CLIPTextModel, CLIPTokenizer


def cargar_pipe():
    """StableDiffusionPipeline armado pieza por pieza (fp16 en la GPU)."""
    piezas = dict(
        vae=AutoencoderKL.from_pretrained(MODELO, subfolder="vae",
                                          torch_dtype=DTYPE),
        text_encoder=CLIPTextModel.from_pretrained(
            MODELO, subfolder="text_encoder", torch_dtype=DTYPE),
        tokenizer=CLIPTokenizer.from_pretrained(MODELO, subfolder="tokenizer"),
        unet=UNet2DConditionModel.from_pretrained(
            MODELO, subfolder="unet", torch_dtype=DTYPE),
        scheduler=DPMSolverMultistepScheduler.from_pretrained(
            MODELO, subfolder="scheduler"))
    pipe = StableDiffusionPipeline(**piezas, safety_checker=None,
                                   feature_extractor=None,
                                   requires_safety_checker=False).to(DEVICE)
    pipe.set_progress_bar_config(disable=True)
    pipe.vae.enable_slicing()          # decodifica de una en una: menos VRAM
    for pieza in (pipe.vae, pipe.text_encoder, pipe.unet):
        pieza.requires_grad_(False)    # nada del modelo base se entrena
    return pipe


PIPE = cargar_pipe()
UNET = PIPE.unet                       # el que recibirá el adaptador LoRA
for nombre in ("unet", "vae", "text_encoder"):
    n = sum(p.numel() for p in getattr(PIPE, nombre).parameters())
    print(f"{nombre:13s}{n / 1e6:8.1f} M parámetros")

`generar(pares, lora, escala)` produce una imagen por pareja `(prompt, semilla)`:

- `lora=False` es el modelo base; `lora=True` usa el adaptador, que existirá después de la sección 05.
- `escala` es la perilla $\lambda$ de la sección 06.
- **Misma semilla ⇒ mismo ruido inicial.** Si el base y el LoRA usan la misma pareja, la diferencia entre las dos imágenes es efecto del adaptador y no del azar.
- `combinar(prompts, semillas)` arma todas las parejas: cada prompt con cada semilla.

In [ ]:
def tiene_lora():
    """¿Ya tiene el UNet un adaptador? (antes de la sección 05, no)."""
    return bool(getattr(UNET, "peft_config", None))


def usar_lora(si):
    """Prende (si=True) o apaga el adaptador. W0 nunca se toca."""
    if tiene_lora():
        (UNET.enable_adapters if si else UNET.disable_adapters)()
    elif si:
        raise RuntimeError("Aún no hay adaptador: corre la sección 05.")


def combinar(prompts, semillas):
    """Todas las parejas (prompt, semilla), prompt por prompt."""
    return [(p, s) for p in prompts for s in semillas]


@torch.no_grad()
def generar(pares, lora=False, escala=1.0):
    """Una imagen por pareja (prompt, semilla). lora=False → el modelo base.
    escala = λ multiplica a ΔW (solo con lora=True)."""
    usar_lora(lora)
    UNET.eval()
    # sin adaptador no se pasa scale: diffusers lo exige solo con LoRA
    extra = {"cross_attention_kwargs": {"scale": escala}} if lora else {}
    salida = []
    for i in range(0, len(pares), 2):  # de 2 en 2: cabe en la T4
        lote = pares[i:i + 2]
        gens = [torch.Generator("cpu").manual_seed(s) for _, s in lote]
        salida += PIPE([p for p, _ in lote], generator=gens,
                       num_inference_steps=P["pasos_inferencia"],
                       height=P["res"], width=P["res"], **extra).images
    return salida

**Cómo leer la malla.** Una fila por prompt de tu `prompts.txt`, una columna por semilla. Esta es la referencia «antes»: en la sección 08 se repite con las **mismas** parejas y el adaptador puesto.

- Fíjate en el estilo, los colores, el encuadre y el fondo. Eso es lo que esperas que cambie.

In [ ]:
ANTES = generar(combinar(PROMPTS, P["semillas"]))
malla(ANTES, filas=len(PROMPTS), titulo="ANTES · el modelo base con tus prompts",
      etiquetas=PROMPTS)                   # una etiqueta por fila

- Arriba, tres imágenes **reales** de validation; abajo, lo que el base genera con **el mismo caption**.
- La distancia entre las dos filas es lo que el fine-tuning intenta cerrar. Si ya se parecen mucho, el cambio después será sutil, porque el base ya sabía hacer tu estilo.

In [ ]:
# Tres imágenes reales de validation (arriba) y lo que el base genera con
# su mismo caption (abajo). Semillas 1000, 1001, … : ninguna otra celda las usa.
k = min(3, len(IMGS["validation"]))
pares_val = [(TEXTOS["validation"][i], 1000 + i) for i in range(k)]
base_val = generar(pares_val)
malla(IMGS["validation"][:k] + base_val, filas=2,
      titulo="validation: real (arriba) y el base con el mismo caption (abajo)",
      etiquetas=[f"real · {t}" for t, _ in pares_val]
      + [f"base · {t}" for t, _ in pares_val])

## 04 · LoRA por dentro

**La idea.** No movemos los pesos del modelo. A cada capa lineal elegida le sumamos una corrección de **rango bajo**:

$$W=W_0+\Delta W,\qquad \Delta W=\frac{\alpha}{r}\,BA,\qquad B\in\mathbb R^{d\times r},\quad A\in\mathbb R^{r\times k},\quad r\ll\min(d,k).$$

| Pieza | Qué es | Se entrena |
|---|---|---|
| $W_0\in\mathbb R^{d\times k}$ | el peso original de la capa | no: congelado |
| $A$, $B$ | las dos matrices delgadas del adaptador | **sí** |
| $r$ | el rango; aquí $r=8$ | — |
| $\alpha$ | una escala; aquí $\alpha=r$, así que $\frac{\alpha}{r}=1$ | — |

- **Empieza sin cambiar nada.** $A$ arranca aleatoria y $B=0$, así que $\Delta W=0$: en el paso 0 el modelo es **exactamente** el base.
- **Rango ≤ $r$.** $\Delta W=BA$ pasa por un espacio de dimensión $r$, así que $\operatorname{rank}(\Delta W)\le r$. Lo verás en la sección 06.
- **Pocos parámetros.** Una capa completa tiene $dk$ números; su adaptador, $r(d+k)$. Con $d=k=768$ y $r=8$: $589{,}824$ contra $12{,}288$, el 2.1 %.
- **Forward barato.** $h=W_0x+\frac{\alpha}{r}B(Ax)$: primero $Ax\in\mathbb R^r$, luego $B(Ax)\in\mathbb R^d$. Nunca se arma la matriz $d\times k$.
- **Perilla.** Al generar se puede usar $W_0+\lambda\,\Delta W$: con $\lambda=0$ vuelve el base, con $\lambda=1$ el adaptado.

**Dónde va en Stable Diffusion.** En las cuatro lineales de cada capa de atención del UNet: `to_q`, `to_k`, `to_v` (las proyecciones de query, key y value) y `to_out.0` (la salida). La celda recorre **tu** UNet, encuentra esas capas y hace la cuenta $dk$ contra $r(d+k)$ con sus tamaños reales.

**Cómo leer la tabla.** Compara la última fila con la segunda: lo que entrenas contra lo que pesa el UNet. Los MB del modelo están en fp16 (2 bytes por número); los del adaptador, en fp32 (4 bytes), como se guarda al final.

In [ ]:
# Las lineales de la atención, como en el notebook 3 y en casi todo LoRA de SD.
MODULOS = ("to_q", "to_k", "to_v", "to_out.0")


def es_objetivo(nombre):
    """La misma regla que peft: el nombre es el módulo o termina en .módulo."""
    return any(nombre == m or nombre.endswith("." + m) for m in MODULOS)


r = P["rango"]
filas = []
for nombre, m in UNET.named_modules():
    nombre = nombre.removesuffix(".base_layer")     # si ya hay adaptador
    if isinstance(m, torch.nn.Linear) and es_objetivo(nombre):
        d, k = m.out_features, m.in_features         # W0 es d × k
        filas.append({"capa": nombre, "d": d, "k": k,
                      "dk": d * k, "r(d+k)": r * (d + k)})
CAPAS = pd.DataFrame(filas).set_index("capa")
N_LORA = int(CAPAS["r(d+k)"].sum())
N_UNET = sum(p.numel() for n, p in UNET.named_parameters() if "lora_" not in n)
N_PIPE = N_UNET + sum(p.numel() for p in PIPE.vae.parameters()) + sum(
    p.numel() for p in PIPE.text_encoder.parameters())

print(f"{len(CAPAS)} capas lineales reciben LoRA. Sus formas (d × k):")
display(CAPAS.groupby(["d", "k"]).size().rename("capas").to_frame())
MB = 1e6
display(pd.DataFrame({
    "parámetros": [N_PIPE, N_UNET, int(CAPAS["dk"].sum()), N_LORA],
    "MB": [2 * N_PIPE / MB, 2 * N_UNET / MB, 2 * CAPAS["dk"].sum() / MB,
           4 * N_LORA / MB],
    "% del UNet": [100 * N_PIPE / N_UNET, 100.0,
                   100 * CAPAS["dk"].sum() / N_UNET, 100 * N_LORA / N_UNET]},
    index=["Stable Diffusion completo (fp16)", "UNet (fp16)",
           "las W0 que se adaptan (fp16)", f"LoRA r={r}: A y B (fp32)"]
).style.format({"parámetros": "{:,.0f}", "MB": "{:,.1f}",
                "% del UNet": "{:.2f} %"}))
print(f"Entrenas {N_LORA:,} números: {100 * N_LORA / N_UNET:.2f} % del UNet.")

## 05 · Entrenar

**Cómo aprende un modelo de difusión.** Se toma el latent $x_0$ de una imagen de train, se le suma ruido, y el UNet tiene que adivinar **qué ruido** se le sumó:

$$x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\varepsilon,\qquad \varepsilon\sim\mathcal N(0,I),\quad t\sim\mathcal U\{0,\dots,T-1\},\ T=1000,$$

$$\mathcal L=\big\lVert\varepsilon-\varepsilon_\theta(x_t,t,c)\big\rVert^2\quad\text{(promedio sobre los elementos del latent)}.$$

- $\bar\alpha_t$ baja de ≈ 1 (casi pura señal) a ≈ 0 (casi puro ruido) conforme crece $t$.
- $c$ es el embedding del **caption de esa imagen**. Así el modelo asocia tu estilo con tus textos.
- Es una **MSE**: regresión, como en la unidad 9. Para generar, se parte de ruido puro y se le resta, paso a paso, el ruido que estima el UNet.

**Precomputar una vez.** El VAE y el text encoder no se entrenan, así que sus salidas no cambian. La celda calcula ahora todos los latents y **un embedding por caption distinto** (si todas tus imágenes dicen lo mismo, es uno solo). Cada paso de entrenamiento solo corre el UNet.

In [ ]:
import time

import torch.nn.functional as F
from diffusers import DDPMScheduler

ESCALA_VAE = PIPE.vae.config.scaling_factor      # 0.18215 en SD 1.5


def a_tensor(imgs):
    """Lista de PIL → tensor (N, 3, H, W) en [-1, 1], como lo espera el VAE."""
    x = torch.stack([torch.from_numpy(np.array(im)) for im in imgs])
    return x.permute(0, 3, 1, 2).float().div(127.5).sub(1)


@torch.no_grad()
def codificar(imgs, lote=8):
    """Latents x_0 = escala · media del VAE, por lotes, guardados en CPU."""
    salida = []
    for i in range(0, len(imgs), lote):
        x = a_tensor(imgs[i:i + lote]).to(DEVICE, DTYPE)
        z = PIPE.vae.encode(x).latent_dist.mode()  # la media: determinista
        salida.append((z * ESCALA_VAE).float().cpu())
    return torch.cat(salida)


LATENTS = {k: codificar(v) for k, v in IMGS.items()}

# Un embedding por caption DISTINTO; CAP[k][i] dice cuál le toca a la imagen i.
CAPTIONS = sorted({t for v in TEXTOS.values() for t in v})
INDICE = {t: i for i, t in enumerate(CAPTIONS)}
CAP = {k: torch.tensor([INDICE[t] for t in v]) for k, v in TEXTOS.items()}
tok = PIPE.tokenizer
with torch.no_grad():
    lotes = []
    for i in range(0, len(CAPTIONS), 32):
        ids = tok(CAPTIONS[i:i + 32], padding="max_length", truncation=True,
                  max_length=tok.model_max_length, return_tensors="pt")
        lotes.append(PIPE.text_encoder(ids.input_ids.to(DEVICE))[0])
    EMBS = torch.cat(lotes).to(DTYPE)            # (captions, 77, D)
largos = sum(len(tok(t).input_ids) > tok.model_max_length for t in CAPTIONS)

for k, z in LATENTS.items():
    print(f"LATENTS[{k!r}]: {tuple(z.shape)}")
print(f"EMBS: {tuple(EMBS.shape)} · {len(CAPTIONS)} captions distintos")
if largos:
    print(f"Ojo: {largos} captions pasan de {tok.model_max_length} tokens; "
          "CLIP solo lee el principio.")

**Una loss de validation que no tiembla.** Si cada medición sorteara $t$ y $\varepsilon$ nuevos, la curva subiría y bajaría por puro azar. Por eso se fijan $K$ pares $(t,\varepsilon)$ con semilla 0, con los $t$ espaciados de 0 a $T-1$, y **todas** las mediciones usan los mismos.

- La celda imprime la loss de validation del **base**. Es el número a vencer.

In [ ]:
RUIDO = DDPMScheduler.from_pretrained(MODELO, subfolder="scheduler")
T = RUIDO.config.num_train_timesteps
gen = torch.Generator().manual_seed(0)
RUIDO_VAL = (torch.linspace(0, T - 1, P["n_val_ruido"]).round().long(),
             torch.randn((P["n_val_ruido"], *LATENTS["train"].shape[1:]),
                         generator=gen))


@torch.no_grad()
def perdida(conjunto, lora=True):
    """MSE del ruido en `conjunto` con (t, ε) FIJOS: la evaluación i usa la
    imagen i mod N y el par i mod K. Dos mediciones usan la misma vara."""
    usar_lora(lora)
    X, C = LATENTS[conjunto], CAP[conjunto]
    t_fijo, eps_fijo = RUIDO_VAL
    n, k = len(X), len(t_fijo)
    estaba = UNET.training
    UNET.eval()
    total, cuenta = 0.0, 0
    for lote in torch.arange(max(n, k)).split(2 * P["batch"]):
        x0 = X[lote % n].to(DEVICE, DTYPE)
        eps = eps_fijo[lote % k].to(DEVICE, DTYPE)
        t = t_fijo[lote % k].to(DEVICE)
        xt = RUIDO.add_noise(x0, eps, t)
        with torch.autocast("cuda", dtype=torch.float16, enabled=CUDA):
            pred = UNET(xt, t, encoder_hidden_states=EMBS[C[lote % n]]).sample
        total += F.mse_loss(pred.float(), eps.float(), reduction="sum").item()
        cuenta += eps.numel()
    UNET.train(estaba)
    return total / cuenta


print("t fijos de validation:", RUIDO_VAL[0].tolist())
print(f"loss de validation del base: {perdida('validation', lora=False):.4f}")

**Un paso de entrenamiento**

1. Toma un batch de latents de train, sortea $\varepsilon$ y $t$, y arma $x_t$.
2. El UNet (con el adaptador) predice $\hat\varepsilon$; la loss es la MSE.
3. `backward` llega **solo** a $A$ y $B$: es lo único con `requires_grad`. AdamW los actualiza.

| Pieza | Valor | Por qué |
|---|---|---|
| Optimizador | AdamW, weight decay $10^{-2}$ | el estándar para LoRA |
| Learning rate | `P["lr"]` $=10^{-4}$ | más alto suele romper la imagen; más bajo casi no mueve nada |
| Schedule | warmup (5 % de los pasos) + cosine | sube suave al principio y baja al final |
| Gradiente | recortado a norma 1 | un batch raro no descompone el adaptador |
| Precisión | UNet fp16, $A$ y $B$ fp32, `autocast` + `GradScaler` | rápido en la T4 sin que los gradientes chicos se vuelvan cero |
| Gradient checkpointing | encendido | guarda menos activaciones y las recalcula: menos memoria |

- La celda de abajo mide los primeros 10 pasos e imprime cuánto va a tardar el entrenamiento completo. Es la celda más larga del notebook.
- Si `GradScaler` encuentra inf o NaN, salta ese paso y baja la escala; ese intento no cuenta.

In [ ]:
from diffusers.optimization import get_scheduler
from diffusers.training_utils import cast_training_params
from peft import LoraConfig

if tiene_lora():                      # si vuelves a correr esta celda
    UNET.delete_adapters("default")
semilla(0)
UNET.add_adapter(LoraConfig(r=P["rango"], lora_alpha=P["rango"],
                            init_lora_weights="gaussian",
                            target_modules=list(MODULOS)))
# A y B en fp32 sobre el UNet fp16: GradScaler no desescala gradientes fp16.
cast_training_params(UNET, dtype=torch.float32)
UNET.enable_gradient_checkpointing()  # menos memoria, un poco más de tiempo
ENTRENABLES = [p for p in UNET.parameters() if p.requires_grad]
N_ENTRENABLES = sum(p.numel() for p in ENTRENABLES)
print(f"parámetros entrenables: {N_ENTRENABLES:,} (la cuenta de la sección 04"
      f" dio {N_LORA:,})")
OPT = torch.optim.AdamW(ENTRENABLES, lr=P["lr"], weight_decay=1e-2)
WARMUP = max(1, round(0.05 * P["pasos"]))
SCHED = get_scheduler("cosine", OPT, num_warmup_steps=WARMUP,
                      num_training_steps=P["pasos"])
ESCALADOR = torch.amp.GradScaler("cuda", enabled=CUDA)
print(f"AdamW lr={P['lr']} · warmup {WARMUP} pasos · cosine hasta "
      f"{P['pasos']} · batch {P['batch']}")

In [ ]:
def un_paso(g):
    """Un paso de difusión sobre un batch de train. Devuelve la loss, o
    None si GradScaler encontró inf/NaN y saltó el paso."""
    X, C = LATENTS["train"], CAP["train"]
    i = torch.randint(len(X), (P["batch"],), generator=g)
    x0 = X[i].to(DEVICE, DTYPE)
    eps = torch.randn(x0.shape, generator=g).to(DEVICE, DTYPE)
    t = torch.randint(0, T, (len(i),), generator=g).to(DEVICE)
    xt = RUIDO.add_noise(x0, eps, t)                   # ensuciar: forward
    with torch.autocast("cuda", dtype=torch.float16, enabled=CUDA):
        pred = UNET(xt, t, encoder_hidden_states=EMBS[C[i]]).sample
    loss = F.mse_loss(pred.float(), eps.float())       # ‖ε − ε̂‖² promedio
    OPT.zero_grad(set_to_none=True)
    ESCALADOR.scale(loss).backward()                   # solo llega a A y B
    ESCALADOR.unscale_(OPT)
    torch.nn.utils.clip_grad_norm_(ENTRENABLES, 1.0)
    escala_previa = ESCALADOR.get_scale()
    ESCALADOR.step(OPT)
    ESCALADOR.update()
    if ESCALADOR.get_scale() < escala_previa:          # inf/NaN: no cuenta
        return None
    SCHED.step()
    return loss.item()


# El punto del paso 0 es el modelo base: B = 0 ⇒ ΔW = 0.
PERDIDAS_TRAIN, PERDIDAS_VAL = [], [(0, perdida("validation"))]
g = torch.Generator().manual_seed(0)
paso, intentos, inicio = 0, 0, time.perf_counter()
usar_lora(True)
UNET.train()
while paso < P["pasos"] and intentos < 2 * P["pasos"]:
    intentos += 1
    loss = un_paso(g)
    if loss is None:
        continue
    paso += 1
    PERDIDAS_TRAIN.append((paso, loss))
    if paso == min(10, P["pasos"]):
        seg = (time.perf_counter() - inicio) / paso
        print(f"{seg:.2f} s/paso → el entrenamiento tardará ≈ "
              f"{seg * P['pasos'] / 60:.1f} min")
    if paso % P["val_cada"] == 0 or paso == P["pasos"]:
        PERDIDAS_VAL.append((paso, perdida("validation")))
        UNET.train()
        print(f"paso {paso:5d} · loss de validation {PERDIDAS_VAL[-1][1]:.4f}")
UNET.eval()
UNET.disable_gradient_checkpointing()
del OPT, SCHED                        # suelta los momentos de AdamW
gc.collect()
if CUDA:
    torch.cuda.empty_cache()
print(f"listo: {paso} pasos en {(time.perf_counter() - inicio) / 60:.1f} min")

**Cómo leer la gráfica.**

- **Train** (azul) brinca muchísimo: cada paso usa otro $t$, y la loss depende sobre todo de $t$. Mira su media móvil, no los puntos.
- **Validation** (naranja) usa siempre los mismos $(t,\varepsilon)$: sus cambios sí son del modelo. El punto gris del paso 0 es el modelo base.
- **Espera cambios chicos.** Una baja pequeña de la loss de validation ya es aprender: buena parte del error de difusión no lo baja ningún modelo, porque el ruido sorteado es azar puro y no hay forma de adivinarlo exacto.
- Si validation **sube** mientras train baja, el adaptador empieza a memorizar train. Con pocos pasos casi nunca pasa; la métrica de memorización de la sección 07 lo mide directo.
- La loss **no es calidad de imagen**. Que baje dice que el modelo predice mejor el ruido de tus imágenes; si las imágenes se ven como tu estilo, lo dicen las mallas y las métricas de CLIP.

In [ ]:
pasos_tr, loss_tr = map(np.array, zip(*PERDIDAS_TRAIN))
ventana = max(1, min(25, len(loss_tr) // 5))
media = np.convolve(loss_tr, np.ones(ventana) / ventana, mode="valid")
fig, eje = plt.subplots(figsize=(8, 3.8))
eje.plot(pasos_tr, loss_tr, color=COLORES["train"], lw=0.7, alpha=0.3,
         label="train (cada paso)")
eje.plot(pasos_tr[ventana - 1:], media, color=COLORES["train"], lw=1.8,
         label=f"train (media de {ventana} pasos)")
eje.plot(*zip(*PERDIDAS_VAL), color=COLORES["validation"], lw=2.5,
         marker="o", ms=4, label="validation (t, ε fijos)")
eje.scatter([0], [PERDIDAS_VAL[0][1]], s=90, color=COLORES["base"], zorder=5,
            label="paso 0 = el modelo base")
eje.set(xlabel="paso", ylabel="MSE del ruido",
        title=f"LoRA r={P['rango']}: loss de train y de validation")
eje.legend(fontsize=8)
plt.tight_layout()
plt.show()
cambio = PERDIDAS_VAL[-1][1] - PERDIDAS_VAL[0][1]
print(f"validation: {PERDIDAS_VAL[0][1]:.4f} (base) → {PERDIDAS_VAL[-1][1]:.4f}"
      f" (LoRA) · cambio {cambio:+.4f} ({100 * cambio / PERDIDAS_VAL[0][1]:+.1f} %)")

## 06 · Ver LoRA por dentro

El adaptador ya está entrenado. Antes de medir si sirve, abrimos la caja: en cada capa adaptada armamos $\Delta W=\frac{\alpha}{r}BA$ con sus $A$ y $B$ reales y lo comparamos con $W_0$.

| Qué miramos | Qué esperamos | Por qué es LoRA y no otra cosa |
|---|---|---|
| (a) valores singulares de $\Delta W$ | caen a ≈ 0 después del $r$-ésimo | $\operatorname{rank}(BA)\le r$ |
| (b) $\lVert\Delta W\rVert_F/\lVert W_0\rVert_F$ por capa | números chicos, desiguales entre capas | el adaptador corrige, no reemplaza |
| (c) la perilla $\lambda$ | $\lambda=0$ da **exactamente** el base | $W_0$ nunca se tocó |

- $\lVert M\rVert_F=\sqrt{\sum_{ij}M_{ij}^2}$ es la norma de Frobenius: el «tamaño» de una matriz.

In [ ]:
from peft.tuners.lora import LoraLayer


def zona(capa):
    """down / mid / up según el nombre de la capa en el UNet."""
    for z, clave in [("down", "down_blocks"), ("mid", "mid_block"),
                     ("up", "up_blocks")]:
        if clave in capa:
            return z
    return "otra"


def tipo(capa):
    """attn1 = self-attention (la imagen consigo misma), attn2 = cross-
    attention (la imagen mira al texto)."""
    return {"attn1": "self-attention", "attn2": "cross-attention"}.get(
        next((p for p in capa.split(".") if p.startswith("attn")), ""), "otra")


filas = []
for nombre, m in UNET.named_modules():
    if isinstance(m, LoraLayer) and "default" in m.lora_A:
        A = m.lora_A["default"].weight.detach().float().cpu()     # r × k
        B = m.lora_B["default"].weight.detach().float().cpu()     # d × r
        dw = m.scaling["default"] * (B @ A)                        # d × k
        w0 = m.base_layer.weight.detach().float().cpu()
        filas.append({"capa": nombre, "zona": zona(nombre),
                      "tipo": tipo(nombre), "dw": dw,
                      "‖ΔW‖/‖W0‖": (dw.norm() / w0.norm()).item()})
DW = pd.DataFrame(filas).set_index("capa")
print(f"{len(DW)} capas con adaptador · ‖ΔW‖/‖W0‖ mediana "
      f"{DW['‖ΔW‖/‖W0‖'].median():.2e}, máxima {DW['‖ΔW‖/‖W0‖'].max():.2e}")
if DW["‖ΔW‖/‖W0‖"].max() == 0:
    print("ΔW = 0 en todas las capas: el adaptador no se movió (¿corrió la 05?)")

**(a) El rango, a la vista.** La SVD escribe $\Delta W=\sum_i\sigma_i u_i v_i^\top$ con $\sigma_1\ge\sigma_2\ge\dots\ge0$. Una matriz de rango $r$ tiene exactamente $r$ valores singulares distintos de cero.

**Cómo leer la gráfica.** El eje vertical es $\sigma_i/\sigma_1$ en escala logarítmica, para la capa que más se movió (➕ en modo largo, también la que más se movió de cada zona). La línea punteada marca $r$. A la izquierda, $r$ valores de tamaño comparable; a la derecha, un salto de muchos órdenes de magnitud hasta el nivel del redondeo de la computadora.

- Aunque $\Delta W$ es una matriz de cientos × cientos, solo vive en $r$ direcciones. Eso es «rango bajo».

In [ ]:
def espectro(capa, ax):
    """σ_i / σ_1 de ΔW en escala log; la línea punteada marca r."""
    s = torch.linalg.svdvals(DW.loc[capa, "dw"]).numpy()
    s = s[: 3 * P["rango"]]
    ax.semilogy(np.arange(1, len(s) + 1), np.maximum(s / max(s[0], 1e-30),
                                                       1e-12), "o-", ms=3,
                label=f"{zona(capa)} · {capa.split('.')[-1]} · {tipo(capa)}")
    return s


fig, ax = plt.subplots(figsize=(7.5, 3.6))
mayor = DW["‖ΔW‖/‖W0‖"].idxmax()            # la capa que más se movió
capas = [mayor]
if LARGO:                                   # ➕ una capa más por zona
    capas += [DW[DW["zona"] == z]["‖ΔW‖/‖W0‖"].idxmax()
              for z in ("down", "mid", "up") if (DW["zona"] == z).any()]
for capa in dict.fromkeys(capas):
    s = espectro(capa, ax)
r = P["rango"]
ax.axvline(r, color="gray", ls="--", lw=1)
ax.text(r + 0.3, 1e-10, f"r = {r}", color="gray")
ax.set(xlabel="índice i", ylabel="σ_i / σ_1", ylim=(1e-12, 3),
       title="Valores singulares de ΔW: después de r, nada")
ax.legend(fontsize=7)
plt.tight_layout()
plt.show()
s = torch.linalg.svdvals(DW.loc[mayor, "dw"])
print(f"{mayor}: ΔW es {tuple(DW.loc[mayor, 'dw'].shape)}; rango numérico "
      f"{torch.linalg.matrix_rank(DW.loc[mayor, 'dw']).item()} (r = {r}); "
      f"σ_{r + 1}/σ_1 = {(s[r] / s[0]).item():.1e}")

**(b) Qué capas cambiaron más.**

- `down` son los bloques que bajan de resolución, `mid` el centro del UNet y `up` los que vuelven a subir.
- **Self-attention** (`attn1`) mezcla partes de la imagen entre sí; **cross-attention** (`attn2`) es donde la imagen mira al texto.

**Cómo leer la gráfica.** Una barra por capa adaptada, en el orden del UNet, coloreada por zona. A la derecha, el promedio por zona y tipo. La tabla lista las cinco capas que más se movieron.

- Todos los cocientes son chicos: el adaptador **corrige** a $W_0$, no lo reemplaza.
- Qué zona gana depende de tu dataset. Un estilo de color y textura suele mover más los bloques de alta resolución; no hay garantía, así que mira tu gráfica.

In [ ]:
COLOR_ZONA = {"down": "#9467bd", "mid": "#8c564b", "up": "#17becf",
              "otra": "gray"}
orden = {"down": 0, "mid": 1, "up": 2, "otra": 3}
tabla_dw = DW.sort_values("zona", key=lambda z: z.map(orden), kind="stable")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 3.8),
                             gridspec_kw={"width_ratios": [2, 1]})
a1.bar(range(len(tabla_dw)), tabla_dw["‖ΔW‖/‖W0‖"],
       color=tabla_dw["zona"].map(COLOR_ZONA))
for z in ("down", "mid", "up"):
    a1.bar([], [], color=COLOR_ZONA[z], label=z)          # solo la leyenda
a1.set(xlabel="capa adaptada (down → mid → up)",
       ylabel=r"$\|\Delta W\|_F\,/\,\|W_0\|_F$",
       title="Cuánto se movió cada capa")
a1.legend()
resumen = tabla_dw.pivot_table(index="zona", columns="tipo",
                               values="‖ΔW‖/‖W0‖", aggfunc="mean")
resumen.reindex([z for z in orden if z in resumen.index]).plot.bar(
    ax=a2, rot=0, title="promedio por zona y tipo")
plt.tight_layout()
plt.show()
display(tabla_dw.nlargest(5, "‖ΔW‖/‖W0‖")[["zona", "tipo", "‖ΔW‖/‖W0‖"]]
        .style.format({"‖ΔW‖/‖W0‖": "{:.2e}"}))

**(c) La perilla $\lambda$.** Al generar se usa $W=W_0+\lambda\,\Delta W$ con el mismo prompt y las mismas semillas:

| $\lambda$ | Qué se aplica |
|---|---|
| 0 | nada: el modelo base |
| 0.5 | la mitad de la corrección |
| 1 | la corrección tal como se entrenó |
| 1.5 | una vez y media: el estilo exagerado |

**Cómo leer la malla.** Una fila por $\lambda$, una columna por semilla. Lee cada columna de arriba abajo: es la misma imagen inicial, cada vez con más adaptador.

- La celda compara $\lambda=0$ con el base (adaptador apagado), píxel por píxel. Debe dar 0 o casi: con $\lambda=0$ el término LoRA vale cero.
- Si entre $\lambda=0$ y $\lambda=1$ casi no hay diferencia, el adaptador aprendió poco. $\lambda=1.5$ muestra hacia dónde iba: es un buen diagnóstico antes de entrenar más.
- Un adaptador también se puede **fusionar** en $W_0$ (`fuse_lora` en diffusers): queda una sola matriz por capa y generar cuesta lo mismo que con el base.

In [ ]:
LAMBDAS = [0.0, 0.5, 1.0, 1.5]
pares = combinar(PROMPTS[:1], P["semillas"][:2])   # el mismo prompt y semillas
imgs = []
for lam in LAMBDAS:
    imgs += generar(pares, lora=True, escala=lam)
malla(imgs, filas=len(LAMBDAS), titulo=f"La perilla λ · «{PROMPTS[0]}»",
      etiquetas=[f"λ = {lam}" for lam in LAMBDAS])

# λ = 0 contra el modelo base (adaptador apagado), píxel por píxel.
base = generar(pares, lora=False)
dif = [np.abs(np.asarray(a, np.float32) - np.asarray(b, np.float32))
       for a, b in zip(imgs[:len(pares)], base)]
print(f"λ = 0 contra el base: diferencia media {np.mean([d.mean() for d in dif]):.3f}"
      f", máxima {max(d.max() for d in dif):.0f} (escala 0–255)")

## 07 · Métricas, explicadas

Mirar mallas convence, pero no mide. Aquí van cinco números; para cada uno, **qué mide, en qué rango vive, qué es bueno y cuál es su trampa**. Todos se calculan en **validation**, que es el conjunto para mirar y decidir.

**Con qué se mide.** Casi todas usan **CLIP** (`openai/clip-vit-base-patch32`), un modelo que lleva imágenes y textos al mismo espacio de vectores: si una imagen y un texto «dicen lo mismo», sus vectores apuntan parecido. Con vectores de norma 1, el parecido es el coseno:

$$\cos(u,v)=u^\top v\in[-1,1].$$

**Qué se genera.** `generadas` toma `P["n_gen"]` captions de validation (en ciclo, si hay menos) y genera una imagen por caption con el base y con LoRA, con las **mismas** parejas (caption, semilla). Así cada diferencia entre las dos columnas es del adaptador.

- Con pocas imágenes los números brincan de una corrida a otra. Lee diferencias grandes, no la tercera cifra decimal.

In [ ]:
from transformers import CLIPModel, CLIPProcessor

CLIP = CLIPModel.from_pretrained(MODELO_CLIP).to(DEVICE).eval()
PROC = CLIPProcessor.from_pretrained(MODELO_CLIP)


def _vector(salida):
    """get_*_features devuelve un tensor o (versiones nuevas) un objeto."""
    if torch.is_tensor(salida):
        return salida
    for campo in ("image_embeds", "text_embeds", "pooler_output"):
        if getattr(salida, campo, None) is not None:
            return getattr(salida, campo)
    raise TypeError(type(salida))


@torch.no_grad()
def emb_imagenes(imgs):
    """(N, D) con norma 1: el coseno entre dos imágenes es su producto punto."""
    out = []
    for i in range(0, len(imgs), 32):
        x = PROC(images=imgs[i:i + 32], return_tensors="pt").to(DEVICE)
        out.append(_vector(CLIP.get_image_features(**x)).float().cpu())
    return F.normalize(torch.cat(out), dim=-1)


@torch.no_grad()
def emb_textos(textos):
    x = PROC(text=list(textos), return_tensors="pt", padding=True,
             truncation=True, max_length=77).to(DEVICE)
    return F.normalize(_vector(CLIP.get_text_features(**x)).float().cpu(), dim=-1)


def clip_score(imgs, textos, emb=None):
    """100 · coseno(imagen_i, texto_i), promediado sobre las parejas."""
    emb = emb_imagenes(imgs) if emb is None else emb
    return float(100 * (emb * emb_textos(textos)).sum(-1).mean())


def generadas(conjunto, semilla0):
    """P["n_gen"] imágenes con los captions de `conjunto` (en ciclo), base y
    LoRA con las MISMAS parejas (caption, semilla), y sus embeddings CLIP."""
    textos = TEXTOS[conjunto]
    pares = [(textos[i % len(textos)], semilla0 + i) for i in range(P["n_gen"])]
    gen = {"textos": [t for t, _ in pares], "emb": {}}
    for k, lora in (("base", False), ("LoRA", True)):
        gen[k] = generar(pares, lora=lora)
        gen["emb"][k] = emb_imagenes(gen[k])
    return gen


EMB_TRAIN = emb_imagenes(IMGS["train"])        # para estilo y memorización
GEN_VAL = generadas("validation", 1000)
print(f"{P['n_gen']} imágenes de validation generadas con el base y con LoRA")

**1 · Loss de difusión (validation)**

| Qué mide | Rango | Qué es bueno | Trampa |
|---|---|---|---|
| qué tan bien predice el UNet el ruido sumado a **tus** imágenes de validation, con $(t,\varepsilon)$ fijos | $\ge 0$ | **más baja** que la del base | **no es calidad visual**: una baja chica puede venir con imágenes muy cambiadas, y al revés. Además depende de qué $t$ se fijaron |

- Es la misma función `perdida` de la sección 05, una vez con el adaptador apagado y otra prendido.

In [ ]:
def m_loss(conjunto):
    """Loss de difusión con (t, ε) fijos, sin y con el adaptador."""
    return {"base": perdida(conjunto, lora=False),
            "LoRA": perdida(conjunto, lora=True)}


VAL = {"loss de difusión": m_loss("validation")}
print(pd.Series(VAL["loss de difusión"]).round(4).to_string())

**2 · CLIP score**

$$\text{CLIP score}=\frac{100}{N}\sum_{i=1}^{N}\cos\big(\text{imagen}_i,\ \text{caption}_i\big).$$

| Qué mide | Rango | Qué es bueno | Trampa |
|---|---|---|---|
| si cada imagen generada **corresponde a su caption**, según CLIP | $[-100,100]$; con este CLIP suele quedar entre 20 y 35 | alto, y que LoRA **no caiga** mucho frente al base | premia «que se vea el objeto», no el estilo: una foto genérica de una cara puede sacar más que tu ilustración. Si tus captions son todos iguales, mide solo eso |

In [ ]:
def m_clip(gen):
    return {k: clip_score(None, gen["textos"], gen["emb"][k])
            for k in ("base", "LoRA")}


VAL["CLIP score"] = m_clip(GEN_VAL)
print(pd.Series(VAL["CLIP score"]).round(2).to_string())

**3 · Parecido al estilo**

Se promedian los vectores CLIP de las imágenes **reales** de validation y se normaliza: es el centroide $\bar e$, «cómo se ve tu dataset». Luego:

$$\text{estilo}=\frac{1}{N}\sum_{i=1}^{N}\cos\big(\text{generada}_i,\ \bar e\big).$$

| Qué mide | Rango | Qué es bueno | Trampa |
|---|---|---|---|
| si las generadas **se parecen a tus imágenes reales** en conjunto | $[-1,1]$; más cerca de 1, más parecido | LoRA **más alto** que el base, acercándose a la referencia | un modelo que copia train saca un valor altísimo. Por eso va junto con la métrica 4 |

- **Referencia:** el mismo número para tus imágenes reales de train. Es «cuánto se parece algo de verdad de tu dataset».

In [ ]:
def m_estilo(conjunto, gen):
    """Coseno medio entre las generadas y el centroide CLIP de las reales
    de `conjunto`. Referencia: las reales de train contra ese centroide."""
    centro = F.normalize(emb_imagenes(IMGS[conjunto]).mean(0), dim=-1)
    salida = {k: float((gen["emb"][k] @ centro).mean()) for k in ("base", "LoRA")}
    salida["referencia"] = float((EMB_TRAIN @ centro).mean())
    return salida


VAL["parecido al estilo"] = m_estilo("validation", GEN_VAL)
print(pd.Series(VAL["parecido al estilo"]).round(3).to_string())

**4 · Memorización: el vecino más cercano en train**

Para cada imagen generada se busca la imagen de **train** más parecida según CLIP, y se guarda ese coseno máximo. Se promedia.

| Qué mide | Rango | Qué es bueno | Trampa |
|---|---|---|---|
| si las generadas son **copias** de imágenes de train | $[-1,1]$; cerca de 1 es casi la misma imagen | **cerca de la referencia** | si tu dataset es muy homogéneo (todas casi iguales), cualquier imagen del estilo tiene un vecino cercano: por eso hay referencia |

- **Referencia:** el mismo número para imágenes **reales de validation**, que el modelo nunca vio. Ese es el parecido «honesto» al vecino: una imagen nueva de tu estilo.
- Si LoRA queda **muy por encima** de la referencia, está repitiendo train. Con pocas imágenes y muchos pasos es lo que pasa.

In [ ]:
def vecino(emb):
    """Coseno CLIP del vecino más cercano en train, para cada fila."""
    return (emb @ EMB_TRAIN.T).max(dim=1).values


def m_memoria(conjunto, gen):
    """Media del coseno al vecino de train: generadas (base, LoRA) y, como
    referencia, imágenes reales de `conjunto` que el modelo nunca vio."""
    salida = {k: float(vecino(gen["emb"][k]).mean()) for k in ("base", "LoRA")}
    salida["referencia"] = float(vecino(emb_imagenes(IMGS[conjunto])).mean())
    return salida


VAL["vecino en train"] = m_memoria("validation", GEN_VAL)
print(pd.Series(VAL["vecino en train"]).round(3).to_string())

**5 · Olvido: prompts ajenos**

Se generan imágenes para tres prompts que no tienen nada que ver con tu dataset, con el base y con LoRA, mismas semillas, y se compara su CLIP score.

| Qué mide | Rango | Qué es bueno | Trampa |
|---|---|---|---|
| si el adaptador **estropea** lo que el modelo ya sabía hacer | como el CLIP score | LoRA ≈ base | un perro «con tu estilo» baja un poco el score sin que el modelo haya olvidado nada: eso es interferencia, no olvido. Olvido es que **el perro deje de verse** |

- LoRA no borra nada de $W_0$: con el adaptador apagado (o $\lambda=0$) vuelve el base tal cual. El «olvido» aquí es lo que pasa **con el adaptador prendido**.

In [ ]:
# Prompts que no tienen nada que ver con el dataset demo. Si tu dataset ES
# de perros, ciudades o fruta, cambia esta lista por otros temas ajenos.
PROMPTS_AJENOS = ["a photo of a dog", "a city street at night",
                  "a bowl of fruit"]


def m_olvido(semillas):
    """CLIP score de cada prompt ajeno, base vs LoRA, mismas semillas.
    Devuelve la tabla por prompt y las imágenes (prompt por prompt)."""
    pares = combinar(PROMPTS_AJENOS, semillas)
    imgs = {"base": generar(pares, lora=False), "LoRA": generar(pares, lora=True)}
    n, filas = len(semillas), []
    for j, prompt in enumerate(PROMPTS_AJENOS):
        filas.append({"prompt": prompt, **{
            k: clip_score(v[j * n:(j + 1) * n], [prompt] * n)
            for k, v in imgs.items()}})
    return pd.DataFrame(filas).set_index("prompt"), imgs


OLVIDO_VAL, _ = m_olvido(P["semillas"])
VAL["olvido"] = OLVIDO_VAL.mean().to_dict()
display(OLVIDO_VAL.assign(**{"LoRA − base": OLVIDO_VAL["LoRA"] -
                             OLVIDO_VAL["base"]}).round(2))

**Todo junto, en validation.** Una fila por métrica, con base, LoRA, la referencia cuando la hay, la diferencia y qué es bueno. Debajo, cuatro imágenes de validation: la real, la del base y la de LoRA con el mismo caption y semilla.

**Cómo leer la tabla.** Ninguna métrica sola dice «salió bien». Un buen LoRA de estilo suele verse así: loss un poco más baja, estilo más alto, CLIP score parecido, vecino en train cerca de la referencia y olvido ≈ base. Si algo no cuadra, la receta de la sección 08 dice qué mover.

In [ ]:
BUENO = {"loss de difusión": "más baja",
         "CLIP score": "alto; que LoRA no caiga mucho",
         "parecido al estilo": "LoRA más alto que base",
         "vecino en train": "cerca de la referencia",
         "olvido": "LoRA ≈ base"}


def resumen(medidas):
    """Una fila por métrica: base, LoRA, referencia y qué es bueno."""
    tabla = pd.DataFrame(medidas).T.reindex(columns=["base", "LoRA",
                                                     "referencia"])
    tabla["LoRA − base"] = tabla["LoRA"] - tabla["base"]
    tabla["qué es bueno"] = pd.Series(BUENO)
    return tabla


display(resumen(VAL).style.format(
    {c: "{:.4f}" for c in ("base", "LoRA", "referencia", "LoRA − base")},
    na_rep="—").set_caption("VALIDATION · base vs LoRA"))
k = min(4, len(IMGS["validation"]), P["n_gen"])
malla(IMGS["validation"][:k] + GEN_VAL["base"][:k] + GEN_VAL["LoRA"][:k],
      filas=3, titulo="validation · real / base / LoRA (mismo caption y semilla)",
      etiquetas=["real", "base", "LoRA"])

## 08 · Testeo, una sola vez

> **Test se usa una vez, aquí, y no se vuelve a tocar.** Si después de ver estos números cambias pasos o learning rate y vuelves a correr, test deja de ser una medida honesta: ya lo usaste para decidir. Para decidir está validation.

- Las mismas cinco métricas de la sección 07, ahora sobre **test**: imágenes que ni el entrenamiento ni tus decisiones vieron.
- Se generan con semillas nuevas (3000 en adelante para los captions de test, 4000 en adelante para los prompts ajenos).
- La segunda tabla pone lado a lado los números de LoRA en validation y en test.

In [ ]:
# Semillas 3000, 3001, … y 4000, …: ninguna sección anterior las usó.
GEN_TEST = generadas("test", 3000)
OLVIDO_TEST, OLVIDO_IMGS = m_olvido([4000 + i for i in range(len(P["semillas"]))])
TEST = {"loss de difusión": m_loss("test"),
        "CLIP score": m_clip(GEN_TEST),
        "parecido al estilo": m_estilo("test", GEN_TEST),
        "vecino en train": m_memoria("test", GEN_TEST),
        "olvido": OLVIDO_TEST.mean().to_dict()}
TABLA_TEST = resumen(TEST)
display(TABLA_TEST.style.format(
    {c: "{:.4f}" for c in ("base", "LoRA", "referencia", "LoRA − base")},
    na_rep="—").set_caption("TEST · base vs LoRA (una sola vez)"))
# Validation contra test, solo para LoRA: ¿fue validation un buen juez?
display(pd.DataFrame({"validation": resumen(VAL)["LoRA"],
                      "test": TABLA_TEST["LoRA"]}).round(4))

**Cómo leer las tablas.**

- **Validation contra test.** Si los números de LoRA se parecen, validation fue un buen juez. Si test sale mucho peor, la partición tuvo mala suerte o el dataset es chico; dilo en tu reporte, no lo corrijas mirando test otra vez.
- **Base contra LoRA.** Lee las cinco filas juntas, como en la sección 07. Un resultado raro (estilo que no sube, loss que sube) también es un resultado: coméntalo con la receta de abajo.

Debajo, dos mallas:

1. **Test:** la imagen real, la del base y la de LoRA con el mismo caption y la misma semilla.
2. **Tus prompts:** para cada prompt, la fila «antes» es la misma de la sección 03 y la fila «después» usa las **mismas semillas** con el adaptador. Compara columna por columna.

In [ ]:
# Test: real / base / LoRA con el mismo caption y semilla.
k = min(8 if LARGO else 4, len(IMGS["test"]), P["n_gen"])
malla(IMGS["test"][:k] + GEN_TEST["base"][:k] + GEN_TEST["LoRA"][:k],
      filas=3, titulo="test · real / base / LoRA (mismo caption y semilla)",
      etiquetas=["real", "base", "LoRA"])

# Tus prompts.txt: las MISMAS parejas de la sección 03, ahora con LoRA.
DESPUES = generar(combinar(PROMPTS, P["semillas"]), lora=True)
n = len(P["semillas"])
imgs, filas = [], []
for j, prompt in enumerate(PROMPTS):
    imgs += ANTES[j * n:(j + 1) * n] + DESPUES[j * n:(j + 1) * n]
    filas += [f"antes · {prompt}", f"después · {prompt}"]
malla(imgs, filas=2 * len(PROMPTS), titulo="Tus prompts: antes y después",
      etiquetas=filas)

**Olvido, a la vista.** Para cada prompt ajeno, una fila con el base y otra con LoRA, mismas semillas.

- Si el perro sigue siendo perro pero «con tu estilo», es **interferencia**: $\Delta W$ actúa en la atención para cualquier prompt.
- Si el perro **deja de verse**, es olvido con el adaptador prendido. Baja $\lambda$ o entrena menos.
- La tabla es el CLIP score de cada prompt; la diferencia `LoRA − base` negativa dice cuánto se alejó del texto.

In [ ]:
n = len(P["semillas"])
imgs, filas = [], []
for j, prompt in enumerate(PROMPTS_AJENOS):
    for quien in ("base", "LoRA"):
        imgs += OLVIDO_IMGS[quien][j * n:(j + 1) * n]
        filas.append(f"{quien} · {prompt}")
malla(imgs, filas=len(filas), titulo="Prompts ajenos: base y LoRA",
      etiquetas=filas)
display(OLVIDO_TEST.assign(**{"LoRA − base": OLVIDO_TEST["LoRA"] -
                              OLVIDO_TEST["base"]}).round(2))

**Tu adaptador, a tu computadora.** Se guardan solo $A$ y $B$ de cada capa, en el formato de diffusers, en `mi_lora/pytorch_lora_weights.safetensors`, y Colab lo descarga. Compara sus MB con los del UNet: eso es LoRA.

- Si el navegador bloquea la descarga, búscalo en el panel 📁 (carpeta `mi_lora`), clic derecho → Descargar.
- Para usarlo después, en cualquier notebook con Stable Diffusion 1.5:

```python
pipe = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5", torch_dtype=torch.float16).to("cuda")
pipe.load_lora_weights("mi_lora")            # la carpeta con el .safetensors
imagen = pipe("an anime face", cross_attention_kwargs={"scale": 1.0}).images[0]
```

In [ ]:
from diffusers.utils import convert_state_dict_to_diffusers
from peft.utils import get_peft_model_state_dict

CARPETA_LORA = Path("mi_lora")
pesos = convert_state_dict_to_diffusers(get_peft_model_state_dict(UNET))
StableDiffusionPipeline.save_lora_weights(CARPETA_LORA, unet_lora_layers=pesos,
                                          safe_serialization=True)
ARCHIVO_LORA = CARPETA_LORA / "pytorch_lora_weights.safetensors"
mb = ARCHIVO_LORA.stat().st_size / 1e6
print(f"{ARCHIVO_LORA}: {mb:.1f} MB (la cuenta de la sección 04 decía "
      f"{4 * N_LORA / 1e6:.1f} MB) · el UNet en fp16: {2 * N_UNET / 1e6:,.0f} MB")
try:                                  # en Colab: lo baja tu navegador
    from google.colab import files
    files.download(str(ARCHIVO_LORA))
except ImportError:
    print("(fuera de Colab: el archivo quedó en", ARCHIVO_LORA.resolve(), ")")

**Si salió mal, qué mover.** Cambia **una** cosa a la vez, vuelve a correr desde la sección 05 y decide con validation.

| Lo que ves | Qué mover |
|---|---|
| LoRA ≈ base: las mallas casi no cambian y el estilo no sube | más `P["pasos"]`; o `P["lr"]` a $2\cdot10^{-4}$. Mira primero $\lambda=1.5$ en la sección 06: si ahí sí se nota, faltan pasos |
| Imágenes rotas, saturadas o ruido; la loss de validation sube | baja `P["lr"]` a $5\cdot10^{-5}$ |
| Vecino en train muy por encima de la referencia; ves copias de train | menos pasos, más imágenes, o `P["rango"]` más chico (4) |
| CLIP score cae mucho: ignora los prompts | captions más descriptivos y variados en `metadata.csv`; menos pasos |
| Los prompts ajenos salen con tu estilo o se pierden | al generar, $\lambda<1$; menos pasos |
| La loss baja pero las imágenes no se ven como tu estilo | normal al principio: la loss no es calidad. Más pasos, o `MODO = "largo"` |
| Error de memoria (*CUDA out of memory*) | `P["batch"]` a 2, y «Entorno de ejecución → Reiniciar» antes de volver a correr |

- **Lo que no es un bug:** con pocas imágenes los números de CLIP cambian si cambias las semillas. Por eso se leen diferencias grandes y se miran las mallas.